# 04 — Modelo probabilístico y de decisión mediante Máxima Utilidad Esperada

**Proyecto Final ADM e IA 2026-2 — Detección de fraude transaccional**  
**Responsable:** Jonathan Chavarría Peralta

## Propósito de esta etapa

El modelo supervisado de Andrés genera una probabilidad estimada de fraude para cada transacción. Sin embargo, una probabilidad por sí sola no determina qué acción conviene tomar.

En este notebook se implementa una capa de decisión basada en **Máxima Utilidad Esperada (MEU)** para transformar la probabilidad de fraude en una recomendación operativa:

- **Aprobar** la transacción.
- **Revisar manualmente** la transacción.
- **Bloquear** la transacción.

La decisión considera la incertidumbre del clasificador y los costos asociados con falsos negativos, falsos positivos y revisiones manuales.

## 1. Relación con el modelo supervisado

El modelo de Regresión Logística obtuvo aproximadamente:

| Métrica | Resultado |
|---|---:|
| Accuracy | 0.8588 |
| Precision | 0.0097 |
| Recall | 0.6875 |
| F1-score | 0.0191 |
| ROC-AUC | 0.8509 |

La matriz de confusión reportó:

- 6,859 transacciones legítimas correctamente identificadas.
- 1,125 transacciones legítimas marcadas como fraude.
- 11 fraudes detectados.
- 5 fraudes no detectados.

El **ROC-AUC cercano a 0.85** indica que el modelo ordena razonablemente bien las transacciones según su riesgo. No obstante, la baja precisión y la gran cantidad de falsos positivos muestran que usar solamente un umbral fijo de 0.5 genera demasiadas alertas.

Por ello, esta etapa no sustituye al clasificador: utiliza su probabilidad para escoger la acción con mayor utilidad esperada.

## 2. Fundamento teórico

Sea \(F\) el evento “la transacción es fraudulenta” y \(\neg F\) el evento “la transacción es legítima”.

Para cada acción \(a\), su utilidad esperada se calcula como:

\[
EU(a)=P(F\mid x)\,U(a,F)+P(\neg F\mid x)\,U(a,\neg F)
\]

donde:

- \(P(F\mid x)\) es la probabilidad estimada de fraude.
- \(P(\neg F\mid x)=1-P(F\mid x)\).
- \(U(a,F)\) es la utilidad de realizar la acción \(a\) cuando sí existe fraude.
- \(U(a,\neg F)\) es la utilidad cuando la operación es legítima.

La acción óptima se obtiene mediante:

\[
a^*=\arg\max_{a\in A}EU(a)
\]

Este criterio permite tomar decisiones considerando simultáneamente probabilidad, costo y beneficio.

## 3. Diseño de la tabla de utilidades

Los valores siguientes son **supuestos académicos del prototipo** y no costos reales de una institución financiera.

Se utiliza el monto de la transacción para que el costo de aprobar un fraude sea mayor cuando la operación es más grande.

| Acción | Si hay fraude | Si es legítima |
|---|---:|---:|
| Aprobar | Pérdida del monto | Ganancia comercial proporcional |
| Revisar | Costo de revisión y riesgo residual | Costo de revisión |
| Bloquear | Evita parte de la pérdida | Pérdida por rechazo incorrecto |

Esta formulación hace que la decisión dependa tanto de la probabilidad de fraude como del monto.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

# Rutas compatibles con ejecución desde la raíz del repositorio o desde notebooks/
ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", ROOT)
print("Carpeta de datos:", DATA_DIR)
print("Carpeta de resultados:", RESULTS_DIR)

## 4. Carga de probabilidades del modelo supervisado

El archivo ideal de entrada es:

```text
results/predicciones_supervisado.csv
```

Debe contener, por lo menos:

- `idx_original`
- `probabilidad_fraude`
- `prediccion_fraude`
- `Class` (cuando esté disponible para evaluación)

La probabilidad es más importante que la predicción binaria, porque la capa MEU necesita medir el grado de riesgo.

In [ ]:
pred_path = RESULTS_DIR / "predicciones_supervisado.csv"

if pred_path.exists():
    predicciones = pd.read_csv(pred_path)
    print("Archivo cargado:", pred_path)
else:
    raise FileNotFoundError(
        "No se encontró results/predicciones_supervisado.csv. "
        "Andrés debe exportar las probabilidades out-of-fold desde su notebook."
    )

columnas_requeridas = {"probabilidad_fraude"}
faltantes = columnas_requeridas - set(predicciones.columns)
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {faltantes}")

predicciones.head()

### Celda que Andrés debe agregar a su notebook

Después de calcular `y_pred` y `y_proba`, debe ejecutar:

```python
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

predicciones_supervisado = pd.DataFrame({
    "idx_original": df["idx_original"],
    "Class": y.to_numpy(),
    "prediccion_fraude": y_pred,
    "probabilidad_fraude": y_proba
})

predicciones_supervisado.to_csv(
    results_dir / "predicciones_supervisado.csv",
    index=False
)
```

Esto evita copiar manualmente resultados y conecta formalmente ambos módulos.

## 5. Incorporación opcional del monto

Si el archivo de predicciones no contiene `Amount`, se intenta recuperarlo desde el conjunto procesado. Si tampoco existe, se utiliza un monto académico constante para permitir ejecutar el prototipo.

En la integración final conviene conservar el monto real desde el preprocesamiento.

In [ ]:
if "Amount" not in predicciones.columns:
    features_path = DATA_DIR / "features_no_supervisado.csv"

    if features_path.exists():
        features = pd.read_csv(features_path)

        if {"idx_original", "Amount"}.issubset(features.columns) and "idx_original" in predicciones.columns:
            predicciones = predicciones.merge(
                features[["idx_original", "Amount"]],
                on="idx_original",
                how="left"
            )

if "Amount" not in predicciones.columns:
    predicciones["Amount"] = 100.0
    print("Advertencia: no se encontró Amount; se usa 100.0 como monto demostrativo.")
else:
    predicciones["Amount"] = pd.to_numeric(predicciones["Amount"], errors="coerce")
    mediana_monto = predicciones["Amount"].median()
    predicciones["Amount"] = predicciones["Amount"].fillna(mediana_monto)

predicciones[["probabilidad_fraude", "Amount"]].describe()

## 6. Funciones de utilidad

Parámetros del prototipo:

- `margen_legitimo`: beneficio proporcional por aprobar una transacción legítima.
- `costo_revision`: costo operativo de revisar manualmente.
- `fraccion_perdida_revision`: parte de la pérdida que aún podría ocurrir durante una revisión.
- `costo_bloqueo_legitimo`: penalización proporcional por bloquear incorrectamente.
- `beneficio_bloqueo_fraude`: fracción de la pérdida evitada al bloquear un fraude.

Los parámetros están centralizados para facilitar análisis de sensibilidad.

In [ ]:
PARAMETROS = {
    "margen_legitimo": 0.02,
    "costo_revision": 8.0,
    "fraccion_perdida_revision": 0.10,
    "costo_bloqueo_legitimo": 0.12,
    "beneficio_bloqueo_fraude": 0.90
}

PARAMETROS

In [ ]:
def calcular_utilidades_esperadas(
    prob_fraude: float,
    monto: float,
    parametros: dict = PARAMETROS
) -> dict:
    """Calcula la utilidad esperada de aprobar, revisar y bloquear."""

    if not 0 <= prob_fraude <= 1:
        raise ValueError("prob_fraude debe estar entre 0 y 1.")
    if monto < 0:
        raise ValueError("monto no puede ser negativo.")

    p_f = prob_fraude
    p_l = 1.0 - p_f

    # Utilidad condicionada al estado real
    u_aprobar_fraude = -monto
    u_aprobar_legitima = parametros["margen_legitimo"] * monto

    u_revisar_fraude = (
        -parametros["costo_revision"]
        - parametros["fraccion_perdida_revision"] * monto
    )
    u_revisar_legitima = -parametros["costo_revision"]

    u_bloquear_fraude = parametros["beneficio_bloqueo_fraude"] * monto
    u_bloquear_legitima = -parametros["costo_bloqueo_legitimo"] * monto

    return {
        "aprobar": p_f * u_aprobar_fraude + p_l * u_aprobar_legitima,
        "revisar": p_f * u_revisar_fraude + p_l * u_revisar_legitima,
        "bloquear": p_f * u_bloquear_fraude + p_l * u_bloquear_legitima
    }


def recomendar_accion(prob_fraude: float, monto: float) -> tuple[str, dict]:
    utilidades = calcular_utilidades_esperadas(prob_fraude, monto)
    accion = max(utilidades, key=utilidades.get)
    return accion, utilidades

## 7. Ejemplo individual explicado

Se prueba una transacción con probabilidad de fraude de 0.70 y monto de 500 unidades monetarias.

In [ ]:
prob_ejemplo = 0.70
monto_ejemplo = 500.0

accion_ejemplo, utilidades_ejemplo = recomendar_accion(
    prob_ejemplo,
    monto_ejemplo
)

print(f"Probabilidad de fraude: {prob_ejemplo:.2%}")
print(f"Monto: {monto_ejemplo:.2f}")
print("\nUtilidades esperadas:")

for accion, utilidad in utilidades_ejemplo.items():
    print(f"  {accion.capitalize():9s}: {utilidad:10.2f}")

print("\nAcción recomendada:", accion_ejemplo.upper())

## 8. Aplicación del modelo de decisión a todas las transacciones

Para cada registro se calculan las tres utilidades y se selecciona la mayor.

In [ ]:
def evaluar_fila(fila: pd.Series) -> pd.Series:
    accion, utilidades = recomendar_accion(
        float(fila["probabilidad_fraude"]),
        float(fila["Amount"])
    )

    return pd.Series({
        "utilidad_aprobar": utilidades["aprobar"],
        "utilidad_revisar": utilidades["revisar"],
        "utilidad_bloquear": utilidades["bloquear"],
        "accion_recomendada": accion
    })


resultado_decision = predicciones.copy()
resultado_decision = pd.concat(
    [resultado_decision, resultado_decision.apply(evaluar_fila, axis=1)],
    axis=1
)

resultado_decision.head()

## 9. Distribución de acciones recomendadas

Esta tabla permite revisar si el sistema está aprobando, revisando o bloqueando una proporción razonable de operaciones.

In [ ]:
distribucion_acciones = (
    resultado_decision["accion_recomendada"]
    .value_counts()
    .rename_axis("accion")
    .reset_index(name="cantidad")
)

distribucion_acciones["porcentaje"] = (
    distribucion_acciones["cantidad"] / len(resultado_decision) * 100
)

distribucion_acciones

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    distribucion_acciones["accion"],
    distribucion_acciones["cantidad"]
)
ax.set_title("Distribución de acciones recomendadas por MEU")
ax.set_xlabel("Acción")
ax.set_ylabel("Número de transacciones")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

ruta_figura_acciones = RESULTS_DIR / "distribucion_acciones_meu.png"
plt.savefig(ruta_figura_acciones, dpi=160, bbox_inches="tight")
plt.show()

print("Figura guardada en:", ruta_figura_acciones)

## 10. Curvas de utilidad y fronteras de decisión

Se analiza cómo cambia la acción óptima al aumentar la probabilidad de fraude para un monto fijo.

Los cruces entre curvas funcionan como umbrales económicos. A diferencia de un umbral arbitrario, estos puntos dependen explícitamente de costos y beneficios.

In [ ]:
probabilidades = np.linspace(0, 1, 501)
monto_referencia = float(resultado_decision["Amount"].median())

curva = []
for p in probabilidades:
    accion, utilidades = recomendar_accion(p, monto_referencia)
    curva.append({
        "probabilidad_fraude": p,
        "utilidad_aprobar": utilidades["aprobar"],
        "utilidad_revisar": utilidades["revisar"],
        "utilidad_bloquear": utilidades["bloquear"],
        "accion_optima": accion
    })

curva_df = pd.DataFrame(curva)
curva_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    curva_df["probabilidad_fraude"],
    curva_df["utilidad_aprobar"],
    label="Aprobar"
)
ax.plot(
    curva_df["probabilidad_fraude"],
    curva_df["utilidad_revisar"],
    label="Revisar"
)
ax.plot(
    curva_df["probabilidad_fraude"],
    curva_df["utilidad_bloquear"],
    label="Bloquear"
)

ax.set_title(
    f"Utilidad esperada por acción — monto de referencia: {monto_referencia:.2f}"
)
ax.set_xlabel("Probabilidad estimada de fraude")
ax.set_ylabel("Utilidad esperada")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()

ruta_curvas = RESULTS_DIR / "curvas_utilidad_meu.png"
plt.savefig(ruta_curvas, dpi=160, bbox_inches="tight")
plt.show()

print("Figura guardada en:", ruta_curvas)

## 11. Estimación de umbrales resultantes

Se identifican los puntos aproximados donde cambia la acción óptima.

In [ ]:
cambios = curva_df[
    curva_df["accion_optima"].ne(curva_df["accion_optima"].shift())
][["probabilidad_fraude", "accion_optima"]].reset_index(drop=True)

print("Cambios aproximados de política de decisión:")
cambios

## 12. Evaluación de la política cuando se conoce la clase real

Si `Class` está disponible, se calcula una matriz de acción contra estado real. Esta tabla no es una matriz de confusión tradicional: evalúa la política operativa final.

In [ ]:
if "Class" in resultado_decision.columns:
    tabla_politica = pd.crosstab(
        resultado_decision["Class"].map({0: "Legítima", 1: "Fraude"}),
        resultado_decision["accion_recomendada"],
        rownames=["Estado real"],
        colnames=["Acción MEU"],
        margins=True
    )
    display(tabla_politica)
else:
    tabla_politica = None
    print("No se encontró Class; se omite la evaluación contra el estado real.")

## 13. Análisis de sensibilidad

Los resultados dependen de los costos asignados. Por ello se comparan varios valores del costo de bloquear una transacción legítima.

Este análisis ayuda a defender que las utilidades no son números arbitrarios invisibles: son parámetros que pueden ajustarse según la política de la organización.

In [ ]:
escenarios = []

for costo_bloqueo in [0.05, 0.12, 0.25]:
    parametros_escenario = PARAMETROS.copy()
    parametros_escenario["costo_bloqueo_legitimo"] = costo_bloqueo

    conteo = {"aprobar": 0, "revisar": 0, "bloquear": 0}

    for _, fila in predicciones.iterrows():
        utilidades = calcular_utilidades_esperadas(
            float(fila["probabilidad_fraude"]),
            float(fila["Amount"]),
            parametros_escenario
        )
        accion = max(utilidades, key=utilidades.get)
        conteo[accion] += 1

    escenarios.append({
        "costo_bloqueo_legitimo": costo_bloqueo,
        **conteo
    })

sensibilidad_df = pd.DataFrame(escenarios)
sensibilidad_df

## 14. Guardado de resultados

Los archivos generados serán consumidos por la aplicación y el reporte final.

In [ ]:
resultado_path = RESULTS_DIR / "resultados_modelo_decision.csv"
tabla_path = RESULTS_DIR / "tabla_utilidades_parametros.csv"
sensibilidad_path = RESULTS_DIR / "sensibilidad_meu.csv"

resultado_decision.to_csv(resultado_path, index=False)
pd.DataFrame([PARAMETROS]).to_csv(tabla_path, index=False)
sensibilidad_df.to_csv(sensibilidad_path, index=False)

print("Resultados guardados:")
print("-", resultado_path)
print("-", tabla_path)
print("-", sensibilidad_path)

## 15. Interpretación de resultados

La Regresión Logística proporciona una probabilidad de fraude con capacidad de discriminación razonable, reflejada en un ROC-AUC cercano a 0.85. Sin embargo, su baja precisión produce numerosas alertas falsas cuando se aplica directamente un umbral de clasificación.

La capa MEU resuelve un problema diferente: no intenta clasificar mejor, sino seleccionar la acción económicamente más conveniente ante la incertidumbre. Una transacción puede tener riesgo moderado y ser enviada a revisión, mientras que una operación de riesgo alto puede bloquearse. Las transacciones de riesgo bajo se aprueban cuando su utilidad esperada supera a las demás alternativas.

Así, el pipeline completo queda conectado de la siguiente manera:

1. El modelo no supervisado genera señales de anomalía y agrupamiento.
2. El modelo supervisado estima la probabilidad de fraude.
3. El modelo de decisión compara las utilidades esperadas.
4. El sistema recomienda aprobar, revisar o bloquear.

La política resultante depende de los costos asumidos. En una implementación real, estos valores deberían estimarse con datos históricos de pérdidas, costos operativos, contracargos, revisiones y rechazo de clientes legítimos.

## 16. Limitaciones

- El conjunto analizado contiene únicamente 16 fraudes, por lo que las probabilidades pueden ser inestables.
- Las utilidades son supuestos académicos y requieren calibración con datos reales.
- El modelo supervisado presenta una precisión muy baja; antes de producción sería necesario ajustar umbrales, calibrar probabilidades y ampliar los datos.
- El monto debe conservarse correctamente desde el preprocesamiento.
- El indicador de anomalía podría incorporarse directamente como evidencia adicional en una futura Red Bayesiana.
- La decisión MEU es explicable, pero depende de que las probabilidades del clasificador estén razonablemente calibradas.

## 17. Conclusión

El modelo de Máxima Utilidad Esperada convierte la salida probabilística del clasificador en una recomendación operativa explícita. Esta capa permite evitar el uso mecánico del umbral 0.5 y considerar las consecuencias distintas de aprobar un fraude, revisar una transacción o bloquear una operación legítima.

El resultado es un componente interpretable, ajustable y adecuado para integrarse en una aplicación interactiva. Su principal aportación al pipeline es separar dos preguntas:

- **¿Qué tan probable es que exista fraude?** — modelo supervisado.
- **¿Qué conviene hacer ante ese nivel de riesgo?** — modelo de decisión.